In [1]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from datasets import Dataset

In [ ]:
MODEL_NAME = "google/muril-base-cased"
MAX_LEN = 96
SEED = 42

# 1.Load data

DATA_DIR = "/home/shivmexe/Projects/grievance_sih/ml/data/grievance_classifier" 
train_pool = pd.read_csv(f"{DATA_DIR}/grievances_synthetic.csv")
hard_holdout = pd.read_csv(f"{DATA_DIR}/grievances_holdout_templates.csv")

le = LabelEncoder()
train_pool["label"] = le.fit_transform(train_pool["category"])
hard_holdout["label"] = le.transform(hard_holdout["category"])

num_labels = len(le.classes_)
print(f"{num_labels} categories: {list(le.classes_)}")

15 categories: ['Banking & Financial Services', 'Corruption & Bribery', 'Education & Schools', 'Electricity', 'Employment & Labour', 'Healthcare & Hospitals', 'Land Records & Revenue', 'Misc / Other', 'Municipal Certificates', 'Pension & Provident Fund', 'Police & Law and Order', 'Ration & Public Distribution System', 'Roads & Infrastructure', 'Sanitation & Garbage', 'Water Supply']


In [3]:
# 2 test train split

train_df, val_df = train_test_split(
    train_pool, test_size=0.15, random_state=SEED, stratify=train_pool["label"]
)

train_ds = Dataset.from_pandas(train_df[["text", "label"]].reset_index(drop=True))
val_ds = Dataset.from_pandas(val_df[["text", "label"]].reset_index(drop=True))
holdout_ds = Dataset.from_pandas(hard_holdout[["text", "label"]].reset_index(drop=True))


In [4]:
# 3 tokenize
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)
holdout_ds = holdout_ds.map(tokenize, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


The OrderedVocab you are attempting to save contains holes for indices [202, 437, 1046, 1057, 1118, 1135, 1150, 1162, 1318, 1445, 1473, 1610, 1626, 1775, 3517, 3643, 4513, 5830, 7834, 12787, 13244, 19712, 25184, 27726, 28024, 31739, 65274], your vocabulary could be corrupted!


Map:   0%|          | 0/2869 [00:00<?, ? examples/s]

The OrderedVocab you are attempting to save contains holes for indices [202, 437, 1046, 1057, 1118, 1135, 1150, 1162, 1318, 1445, 1473, 1610, 1626, 1775, 3517, 3643, 4513, 5830, 7834, 12787, 13244, 19712, 25184, 27726, 28024, 31739, 65274], your vocabulary could be corrupted!


Map:   0%|          | 0/507 [00:00<?, ? examples/s]

The OrderedVocab you are attempting to save contains holes for indices [202, 437, 1046, 1057, 1118, 1135, 1150, 1162, 1318, 1445, 1473, 1610, 1626, 1775, 3517, 3643, 4513, 5830, 7834, 12787, 13244, 19712, 25184, 27726, 28024, 31739, 65274], your vocabulary could be corrupted!


Map:   0%|          | 0/664 [00:00<?, ? examples/s]

In [5]:
# 4 model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=num_labels
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }

args = TrainingArguments(
    output_dir="./grievance_muril_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=6,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    save_total_limit=1,
    logging_steps=25,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params w

AcceleratorError: CUDA error: out of memory
Search for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
For more detailed error information, run with CUDA_LOG_FILE=stderr


In [ ]:
# 5. Train

trainer.train()


In [ ]:
# 6. Evaluation

print("\n=== Validation set ===")
print(trainer.evaluate(val_ds))

print("\n=== Holdout set ===")
holdout_metrics = trainer.evaluate(holdout_ds)
print(holdout_metrics)

preds = trainer.predict(holdout_ds)
pred_labels = np.argmax(preds.predictions, axis=-1)
print(classification_report(
    hard_holdout["label"], pred_labels, target_names=le.classes_, digits=3
))


In [ ]:
# 7. Save locally, then push to Hugging Face Hub
trainer.save_model("home/shivmexe/Projects/grievance_sih/ml/models/large_model")
tokenizer.save_pretrained("home/shivmexe/Projects/grievance_sih/ml/models/large_model")

import json
with open("home/shivmexe/Projects/grievance_sih/ml/models/large_model/label_map.json", "w", encoding="utf-8") as f:
    json.dump({str(i): c for i, c in enumerate(le.classes_)}, f, ensure_ascii=False, indent=2)

print("Model saved to ./grievance_muril_model_final")

# --- Uncomment to push directly to Hugging Face Hub from Kaggle ---
# from huggingface_hub import login
# login(token="YOUR_HF_TOKEN")  # set as a Kaggle Secret, don't hardcode
# trainer.push_to_hub("your-username/citizen-grievance-classifier-muril")
# tokenizer.push_to_hub("your-username/citizen-grievance-classifier-muril")

In [ ]:
from transformers import AutoTokenizer
import torch
import json

with open("home/shivmexe/Projects/grievance_sih/ml/models/large_model/label_map.json", "r", encoding="utf-8") as f:
    label_map = json.load(f)
id2label = {int(k): v for k, v in label_map.items()}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Prediction function
def predict_grievance(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=96, padding=True).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)
        pred_id = probs.argmax(dim=-1).item()
        confidence = probs[0][pred_id].item()
    return id2label[pred_id], confidence

# --- TEST ON NEW UNSEEN DATA ---
new_complaints = [
    "Sir, pichle 10 din se hamare mohalle ka streetlight band hai, raat ko chori hone ka dar hai.",
    "Mera ration card 1 mahine se pending hai, dealer anaj nahi de raha.",
    "विद्यालय में छात्रवृत्ति का पैसा अभी तक नहीं आया है।",
    "Sadak par aawara kutte bahut hain, kal ek bachche ko kaat liya.",
    "Mera mobile network 2 hafte se kaam nahi kar raha is area mein.",
    "sadak ke ghade ho rhe he mere yaha",
    "mere vidyalay me shikshoka ki kami he",
    "mere yaha gunda gardi bad rhi he",
    "bijli kab aayegi",
    "aaj kal baarish nhi ho rhi he",
]

print("\n--- Predictions ---")
for complaint in new_complaints:
    category, confidence = predict_grievance(complaint)
    print(f"Text: {complaint}")
    print(f"Predicted Category: {category} (Confidence: {confidence*100:.2f}%)\n")